<a href="https://colab.research.google.com/github/sarahibdah/Flyrank-ML-Internship-Sarah/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarahibdah/Flyrank-ML-Internship-Sarah/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import precision_score, recall_score, accuracy_score

url = "https://raw.githubusercontent.com/sarahibdah/Flyrank-ML-Internship-Sarah/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
y = (df["trend_direction"] == "down").astype(int)   # same label as Week 5
g = df["client_id"]
base_rate = y.mean()
print(f"{len(df):,} pages, {g.nunique()} clients, base rate (share declining) = {base_rate:.3f}")


30,000 pages, 32 clients, base rate (share declining) = 0.542


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

I read *The State of AI-Driven SEO in Numbers* (March 2026). The paper is upfront about its limits: it calls itself observational, says "correlations do not prove causation", and labels its ML pages exploratory. The questions below are about how I would make two of its findings even stronger, the way I want my own work reviewed.

**Finding A: the refresh result.** *365+ day content refreshed within 30 days shows a 3.2x health boost (10.7 to 34.5) and 57x more impressions (71 to 4,039); refresh timing is "one of the strongest measured levers".*

- **Where does the comparison come from?** The paper compares refreshed pages with unrefreshed ones. My question: who decided which pages got refreshed? If editors chose pages that were already healthy or already recovering, part of the gap comes from the choosing, not from the refresh. A matched comparison (same age, same starting traffic) or a before/after on the same pages would separate the two.
- **Is the outcome independent of the grouping?** Health score includes impressions (30 points), so a big impressions gap partly produces a big health gap by construction. I would report impressions and clicks on their own, with the group sizes (n), next to the composite.
- **Does the wording fit the design?** "Lever" reads as causal. With the paper's own observational design, "pages refreshed within 30 days showed higher impressions" carries the same evidence in words the design supports.

**Finding B: the ML appendix, "What Predicts Health?"** *Random Forest feature importance for health score: average position 43, impressions 32, scroll depth 15.*

- **Where does the label come from?** The paper says health score is built from impressions (30 pts), position (30 pts), CTR (20 pts) and scroll depth (20 pts). The top three features are therefore ingredients of the label, which is why the paper notes that importance is descriptive. I would take the next step and name this as label-derived leakage (my Week-3 lesson), or predict something not built from the inputs, such as next-period traffic change.
- **Does the validation design support the claim?** The paper uses an 80/20 split. My question: is that split random across pages, or grouped by brand? With 57 brands, pages from one brand share hidden traits, and a random split can let a model look better than it will on a new brand. Reporting a grouped split next to the random one would show how much memorisation there is.
- **What is the base rate?** For the logistic-regression result (71% holdout accuracy), I would print the share of growing pages next to it, so readers see how many points are skill.

I applied the same checks to my own Week-5 model in the sections below, and my own claims needed the most fixing.


In [2]:
# Section 1 is written analysis. This cell only restates the paper figures I am asking about,
# so the questions above sit next to the numbers they refer to.
paper_figures = pd.DataFrame({
    "finding": ["A: refresh (365+ days)", "A: refresh (365+ days)", "B: RF importance", "B: RF importance", "B: RF importance"],
    "figure_in_paper": ["health 10.7 -> 34.5 (3.2x)", "impressions 71 -> 4,039 (57x)",
                        "average position = 43", "impressions = 32", "scroll depth = 15"],
    "question": ["who chose which pages were refreshed?", "impressions are also inside health score",
                 "position is an ingredient of the label", "impressions are an ingredient of the label",
                 "scroll depth is an ingredient of the label"],
})
paper_figures


,finding,figure_in_paper,question
0,A: refresh (365+ days),health 10.7 -> 34.5 (3.2x),who chose which pages were refreshed?
1,A: refresh (365+ days),"impressions 71 -> 4,039 (57x)",impressions are also inside health score
2,B: RF importance,average position = 43,position is an ingredient of the label
3,B: RF importance,impressions = 32,impressions are an ingredient of the label
4,B: RF importance,scroll depth = 15,scroll depth is an ingredient of the label


## 2. My model under an honest split (before/after)

*Re-run your Week-3 model under a grouped or time-aware split. Show both numbers.*

**Before (Week 5):** one grouped split, 25 clients to train and 7 to test. It reported tree precision 0.623 and recall 0.916 against the age rule's 0.627 and 0.556, and I called that a "genuine, meaningful improvement".

**After (this notebook):** I rebuilt the same two-feature tree (`age_tier_order`, `impressions_90d`, depth 4) and the same age rule, then ran them (1) under a **random** 5-fold split, (2) under a **client-grouped** 5-fold split (`GroupKFold`), and (3) fold by fold. I also print the base rate and the "flag everything" row, because a method that flags more pages will show higher recall without being better.


In [3]:
def rule_flags(d):                       # Week-4 style age rule: tiers 3 and 4 (31-180 days)
    return d["age_tier_order"].isin([3, 4]).astype(int)

def score(y_true, pred):
    return dict(accuracy=accuracy_score(y_true, pred), precision=precision_score(y_true, pred),
                recall=recall_score(y_true, pred), flag_rate=float(np.mean(pred)))

def cv_tree(cols, splitter, **kw):
    pred = pd.Series(0, index=df.index)
    for tr, te in splitter.split(df, y, g):
        m = DecisionTreeClassifier(max_depth=4, random_state=42, **kw).fit(df.iloc[tr][cols], y.iloc[tr])
        pred.iloc[te] = m.predict(df.iloc[te][cols])
    return pred

feat = ["age_tier_order", "impressions_90d"]
rand_pred = cv_tree(feat, KFold(5, shuffle=True, random_state=42))
grp_pred  = cv_tree(feat, GroupKFold(5))

table = pd.DataFrame({
    "Tree, random 5-fold":          score(y, rand_pred),
    "Tree, GroupKFold by client":   score(y, grp_pred),
    "Age rule (no training)":       score(y, rule_flags(df)),
    "Flag every page":              score(y, np.ones(len(df), dtype=int)),
}).T.round(3)
print(f"Base rate (share declining) = {base_rate:.3f}")
table


Base rate (share declining) = 0.542


,accuracy,precision,recall,flag_rate
"Tree, random 5-fold",0.644,0.645,0.763,0.641
"Tree, GroupKFold by client",0.639,0.636,0.777,0.662
Age rule (no training),0.562,0.627,0.473,0.409
Flag every page,0.542,0.542,1.000,1.000


In [4]:
# Fold by fold: does the Week-5 conclusion hold on every group of unseen clients?
rows = []
for k, (tr, te) in enumerate(GroupKFold(5).split(df, y, g)):
    m = DecisionTreeClassifier(max_depth=4, random_state=42).fit(df.iloc[tr][feat], y.iloc[tr])
    p = m.predict(df.iloc[te][feat]); b = rule_flags(df.iloc[te]); yt = y.iloc[te]
    rows.append(dict(fold=k, n_pages=len(te), base_rate=yt.mean(),
                     tree_precision=precision_score(yt, p), rule_precision=precision_score(yt, b),
                     tree_recall=recall_score(yt, p), rule_recall=recall_score(yt, b),
                     tree_flag_rate=p.mean(), rule_flag_rate=b.mean()))
folds = pd.DataFrame(rows).round(3)
folds["tree_lift_over_base"] = (folds.tree_precision / folds.base_rate).round(2)
folds["rule_lift_over_base"] = (folds.rule_precision / folds.base_rate).round(2)
print(folds.to_string(index=False))
print("\nMean over folds: tree precision %.3f vs rule precision %.3f | tree recall %.3f vs rule recall %.3f"
      % (folds.tree_precision.mean(), folds.rule_precision.mean(), folds.tree_recall.mean(), folds.rule_recall.mean()))


 fold  n_pages  base_rate  tree_precision  rule_precision  tree_recall  rule_recall  tree_flag_rate  rule_flag_rate  tree_lift_over_base  rule_lift_over_base
    0     7008      0.490           0.524           0.628        0.747        0.261           0.698           0.204                 1.07                 1.28
    1     5731      0.645           0.652           0.649        0.924        0.559           0.914           0.556                 1.01                 1.01
    2     5753      0.379           0.541           0.418        0.474        0.313           0.333           0.284                 1.43                 1.10
    3     5755      0.622           0.724           0.683        0.939        0.606           0.807           0.552                 1.16                 1.10
    4     5753      0.585           0.713           0.660        0.673        0.558           0.551           0.494                 1.22                 1.13

Mean over folds: tree precision 0.631 vs rule preci

**What the before/after shows** (numbers are in the tables above):

- **Random vs grouped barely differ** for these two features. Precision moves by under one point, so I saw no sign that the tree was memorising clients. The honest split matters for the claim even though it did not change the number here.
- **Precision is modestly above the base rate.** The grouped-split tree reaches precision 0.636, about 9 points above the 0.542 share of declining pages (lift about 1.17). "Flag every page" already gets 0.542 precision with recall 1.0, so recall alone cannot show skill.
- **The Week-5 recall gap is mostly a flag-rate gap.** Across folds the tree flags between about a third and nine tenths of pages while the age rule flags a fifth to a half. Flagging more pages raises recall by itself, so recall 0.916 vs 0.556 does not show a better ranking.
- **Which clients are held out changes the picture.** The base rate per fold runs from about 0.38 to 0.65. The age rule has the higher precision in fold 0 (0.628 vs 0.524), the two tie in fold 1, and the tree is ahead in folds 2 to 4. On average over folds the tree's precision is 0.631 against the rule's 0.608. The Week-5 result rested on one draw of 7 clients.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I drew the timeline first. The label compares **impressions in the last 30 days** with **impressions in the 30 days before that** (`trend_direction`, from `trend_pct`). A legal feature must be knowable before the last-30 window starts: `age_tier_order`, `impressions_prev_30d` (days 31-60 back) and the older part of the 90-day total (days 61-90). `impressions_90d` is the sum of all three windows, so it **contains the label window**.

| Feature | Window | Verdict |
|---|---|---|
| `age_tier_order` | page age, from `content_age_days` | OK: knowable at the decision moment |
| `impressions_90d` | days 0-90, includes the label's last-30 window | **Leak risk (overlapping window)**; replace with legal sub-windows |
| `impressions_prev_30d` | days 31-60 back | OK as a feature (it is the label's denominator, but it is earlier than the last-30 window) |
| `trend_direction`, `trend_pct` | the label itself | Never a feature |


In [5]:
# 1) How much of impressions_90d is the label window?
df["imp_61_90"] = df["impressions_90d"] - df["impressions_last_30d"] - df["impressions_prev_30d"]
print("min of (90d - last30 - prev30):", df["imp_61_90"].min(), "(never negative: the 90d total contains both windows)")
print("corr(impressions_90d, impressions_last_30d) = %.3f" % df["impressions_90d"].corr(df["impressions_last_30d"]))
print("median share of impressions_90d that is the last-30 window: %.2f"
      % (df["impressions_last_30d"] / df["impressions_90d"]).median())

# 2) Train WITH and WITHOUT the suspect, grouped split, base rate printed
df["leak_trend_pct"] = df["trend_pct"]
legal = ["age_tier_order", "impressions_prev_30d", "imp_61_90"]
sets = {
    "legal: age + prev30 + days 61-90":        legal,
    "suspect: age + impressions_90d (Week 5)": ["age_tier_order", "impressions_90d"],
    "legal + impressions_90d":                 legal + ["impressions_90d"],
    "legal, prev30 only: age + prev30":        ["age_tier_order", "impressions_prev_30d"],
    "age only":                                ["age_tier_order"],
    "DELIBERATE LEAK: legal + trend_pct":      legal + ["leak_trend_pct"],
}
res = {name: score(y, cv_tree(cols, GroupKFold(5), min_samples_leaf=50)) for name, cols in sets.items()}
audit = pd.DataFrame(res).T.round(3)
print(f"\nBase rate = {base_rate:.3f} (majority-class accuracy = {max(base_rate, 1-base_rate):.3f})")
audit


min of (90d - last30 - prev30): 0 (never negative: the 90d total contains both windows)
corr(impressions_90d, impressions_last_30d) = 0.918
median share of impressions_90d that is the last-30 window: 0.22

Base rate = 0.542 (majority-class accuracy = 0.542)


,accuracy,precision,recall,flag_rate
legal: age + prev30 + days 61-90,0.668,0.645,0.861,0.724
suspect: age + impressions_90d (Week 5),0.639,0.636,0.779,0.663
legal + impressions_90d,0.673,0.644,0.888,0.748
"legal, prev30 only: age + prev30",0.668,0.645,0.862,0.724
age only,0.552,0.566,0.739,0.707
DELIBERATE LEAK: legal + trend_pct,1.000,1.000,1.000,0.542


**Audit result.**

- The harness works: adding the label's own source column (`trend_pct`) takes the score to 1.000, so the leak test would have caught a serious leak. I removed it and keep the legal-feature number.
- `impressions_90d` overlaps the label window (correlation with last-30 impressions 0.92), so it fails the timeline test. Adding it on top of the legal features raised accuracy by only about half a point (0.668 to 0.673), so I did not measure a large inflated score from it in this dataset. The timeline still matters, because for a future page the last-30 window does not exist yet.
- The legal set (age + prev30 + days 61-90) scores about 3 points higher than my Week-5 pair (0.668 vs 0.639) and about 12 points above the 0.542 base rate. Age alone is only about 1 point above the base rate (0.552).
- Age + prev30 alone gives the same accuracy as the full legal set (0.668), so in this snapshot the gain over age-only comes from `impressions_prev_30d`. That column is the denominator of the label, so I read the model as a short-horizon momentum signal, not a lifecycle model. This is directional, from one snapshot.
- My Week-4 note said `impressions_90d` was "not a future-window value". That was wrong for this label, and Section 4 rewrites it.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

| # | My original claim | What is wrong with it | Rewrite |
|---|---|---|---|
| 1 | Week 5: "This is a genuine, meaningful improvement... the tree combines two validated signals." | Precision was tied (0.623 vs 0.627). The recall gap tracks flag rate (tree flags far more pages), the result used one 7-client split, and the per-fold table shows the gap varies. | "On one held-out set of 7 clients I **observed** similar precision (0.62 vs 0.63) and higher recall for the tree, which also flagged more pages. Across 5 grouped folds the tree's precision averaged 0.631 vs 0.608 for the rule (about 9 points above the 0.542 base rate for the tree), with the rule ahead in one fold. I treat this as **directional**." |
| 2 | Week 5: "The model beats the baseline... by using more of the evidence I'd already gathered and validated." | "Validated" overstates it: Signal 2 was MIXED, and one of the two features overlaps the label window. | "The tree used age and impressions; the evidence for impressions as a risk signal was **mixed**, and the feature overlaps the label window (Section 3)." |
| 3 | Week 4: "`impressions_90d`: trailing historical metric, not a future-window value." | Incorrect. The 90-day total includes the last-30 and previous-30 windows the label compares. | "`impressions_90d` **contains** the label window, so I rebuilt features from `impressions_prev_30d` and days 61-90 for the audit." |
| 4 | Week 5: "The tree over-generalizes the pattern from my Signal 2 finding." | I did not test that cause. | "False positives were mostly pages whose traffic was **stable or up** (55% stable and 38% up among the wrongly flagged pages). One possible reading is that large pages with flat traffic look like decliners to a two-feature tree. I have not tested that." |
| 5 | Week 4 top-20: "genuine save" / "refresh_now" ranking as the editor's list. | The ranking was never scored out-of-sample. | "As **decision-support**, these pages look worth reviewing first. I have not measured whether refreshing them changes outcomes." |

**Decision-support statement I will stand behind.** *In this snapshot of 30,000 pages from 32 clients, a two-feature tree flagged more pages than an age-only rule, with precision about 2 points higher on average across grouped folds (0.631 vs 0.608) and about 9 points above the base rate. The result is directional, rests on one snapshot, and does not show that refreshing flagged pages improves traffic.*


In [6]:
# Real failure examples behind rewrite #4 and the Week-4 queue.
X = df[feat]; pred = grp_pred
fp = df[(pred == 1) & (y == 0)]
fn = df[(pred == 0) & (y == 1)]
print(f"False positives: {len(fp):,} | False negatives: {len(fn):,}")
print("\nWhat the false positives actually did (share of the wrongly flagged):")
print(fp["trend_direction"].value_counts(normalize=True).round(3).to_string())

cols = ["content_id", "age_tier", "impressions_90d", "impressions_prev_30d", "impressions_last_30d", "trend_direction"]
print("\nLargest false positives (flagged as declining, but traffic held or grew):")
print(fp.sort_values("impressions_90d", ascending=False).head(5)[cols].to_string(index=False))
print("\nLargest false negatives (real decliners the tree missed):")
print(fn.sort_values("impressions_90d", ascending=False).head(5)[cols].to_string(index=False))
fn_top = fn.sort_values("impressions_90d", ascending=False).head(5)
print("\nTop-5 false negatives: age tiers =", fn_top["age_tier"].tolist(),
      "| drop vs prev 30d =", ((fn_top.impressions_last_30d / fn_top.impressions_prev_30d - 1) * 100).round(0).tolist())


False positives: 7,221 | False negatives: 3,621

What the false positives actually did (share of the wrongly flagged):
trend_direction
stable    0.553
up        0.380
new       0.042
flat      0.025

Largest false positives (flagged as declining, but traffic held or grew):
          content_id age_tier  impressions_90d  impressions_prev_30d  impressions_last_30d trend_direction
content_aaef01a50def     365+           517109                177601                170559          stable
content_2cb567c3c89b   91-180           497727                160641                197677              up
content_2dba2b1f9536  181-365           443434                137909                139891          stable
content_44e481c8f55b   91-180           312694                118137                104458          stable
content_36ff89c8214e   91-180           295097                106412                106985          stable

Largest false negatives (real decliners the tree missed):
          content_id age_

**What the failure examples show (observed in this snapshot).** Four of the five largest missed decliners are 365+ day pages with 300,000 to 500,000 impressions that lost 33% to 45% of their traffic; the fifth lost 89%. Signal 1 said the 365+ tier declines *least often*, so a model built around decline *rate* can miss the biggest decline *losses*. This is a pattern in 5 examples, not a tested result, and I list it as a question for my capstone. The largest false positives are big pages whose traffic was stable or up.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.